In [4]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [6]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

print("Collection ready:", collection.name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection ready: rag_chunks


Exercise 1

In [7]:
# Third document

third_doc_text = """
Large Language Models and RAG

Large Language Models, or LLMs, are AI models trained on large amounts
of text data. They can understand and generate human-like text.

RAG improves the reliability of LLM applications by retrieving relevant
information from an external knowledge base before generating an answer.

The retrieved information is provided as context to the language model,
helping it produce answers that are more accurate and grounded in the
available documents.
"""

# Split into chunks
third_doc_chunks = splitter.split_text(third_doc_text)

# Metadata
third_doc_metadatas = [
    {
        "source": "llm_and_rag.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(third_doc_chunks)
]

# IDs
third_doc_ids = [
    f"llm_and_rag_chunk_{i}"
    for i in range(len(third_doc_chunks))
]

# Store in ChromaDB
collection.add(
    documents=third_doc_chunks,
    metadatas=third_doc_metadatas,
    ids=third_doc_ids
)

print("Third document chunks:", len(third_doc_chunks))
print("Total chunks in ChromaDB:", collection.count())

Third document chunks: 3
Total chunks in ChromaDB: 3


Exercise 2 — Re-chunk with chunk_size=150

In [13]:
# ==============================
# EXERCISE 2
# Re-chunk ai_overview.txt with chunk_size=150
# ==============================

# Original document
article_text = """
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
"""

# Original splitter: 300 characters
original_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

original_chunks = original_splitter.split_text(article_text)

# New splitter: 150 characters
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

small_chunks = small_splitter.split_text(article_text)

print("Original number of chunks:", len(original_chunks))
print("New number of chunks:", len(small_chunks))


# --------------------------------
# Create metadata
# --------------------------------

small_metadatas = [
    {
        "source": "ai_overview_v2.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(small_chunks)
]

# Create new IDs
small_ids = [
    f"ai_overview_v2_chunk_{i}"
    for i in range(len(small_chunks))
]

print("\nExample ID:", small_ids[0])
print("Example metadata:", small_metadatas[0])


# --------------------------------
# Store new chunks in ChromaDB
# --------------------------------

collection.add(
    documents=small_chunks,
    metadatas=small_metadatas,
    ids=small_ids
)

print("\nNew chunks added:", len(small_chunks))
print("Total chunks in ChromaDB:", collection.count())


# --------------------------------
# Retrieval from original chunks
# --------------------------------

original_results = collection.query(
    query_texts=["What is deep learning?"],
    n_results=3,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

print("\n" + "=" * 60)
print("ORIGINAL 300-CHARACTER CHUNKS")
print("=" * 60)

for doc, meta, distance in zip(
    original_results["documents"][0],
    original_results["metadatas"][0],
    original_results["distances"][0]
):
    print("Distance:", round(distance, 4))
    print("Chunk:", meta["chunk_index"])
    print(doc)
    print("-" * 60)


# --------------------------------
# Retrieval from new chunks
# --------------------------------

new_results = collection.query(
    query_texts=["What is deep learning?"],
    n_results=3,
    where={"source": "ai_overview_v2.txt"},
    include=["documents", "metadatas", "distances"]
)

print("\n" + "=" * 60)
print("NEW 150-CHARACTER CHUNKS")
print("=" * 60)

for doc, meta, distance in zip(
    new_results["documents"][0],
    new_results["metadatas"][0],
    new_results["distances"][0]
):
    print("Distance:", round(distance, 4))
    print("Chunk:", meta["chunk_index"])
    print(doc)
    print("-" * 60)

Original number of chunks: 9
New number of chunks: 19

Example ID: ai_overview_v2_chunk_0
Example metadata: {'source': 'ai_overview_v2.txt', 'chunk_index': 0, 'char_count': 36}

New chunks added: 19
Total chunks in ChromaDB: 22

ORIGINAL 300-CHARACTER CHUNKS

NEW 150-CHARACTER CHUNKS
Distance: 0.3418
Chunk: 7
Deep Learning takes Machine Learning further by using neural networks with many layers. These
------------------------------------------------------------
Distance: 0.5322
Chunk: 4
Machine Learning is a subset of AI where systems learn patterns from data instead of following
------------------------------------------------------------
Distance: 0.6168
Chunk: 1
Artificial Intelligence, or AI, is the field of computer science focused on building systems
------------------------------------------------------------


Exercise 3  Create add_document()

In [14]:
# ==============================
# EXERCISE 3
# Create reusable add_document()
# ==============================

def add_document(collection, splitter, text, source_name):

    # 1. Split document into chunks
    document_chunks = splitter.split_text(text)

    # 2. Create metadata
    document_metadatas = [
        {
            "source": source_name,
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(document_chunks)
    ]

    # 3. Create unique IDs
    document_ids = [
        f"{source_name.replace('.txt', '')}_chunk_{i}"
        for i in range(len(document_chunks))
    ]

    # 4. Store in ChromaDB
    collection.add(
        documents=document_chunks,
        metadatas=document_metadatas,
        ids=document_ids
    )

    # 5. Return number of chunks
    return len(document_chunks)


# --------------------------------
# Test the function
# --------------------------------

new_document = """
Python is a high-level programming language known for its simple syntax.

Python is widely used in web development, data science, artificial
intelligence, machine learning and automation.

Popular Python libraries include NumPy, Pandas, Scikit-learn and TensorFlow.
"""

chunks_added = add_document(
    collection,
    splitter,
    new_document,
    "python_basics.txt"
)

print("Chunks added:", chunks_added)
print("Total chunks in ChromaDB:", collection.count())


# --------------------------------
# Verify the document
# --------------------------------

results = collection.get(
    where={"source": "python_basics.txt"},
    include=["documents", "metadatas"]
)

print("\nNumber of Python document chunks:", len(results["documents"]))

for doc, meta in zip(
    results["documents"],
    results["metadatas"]
):
    print("\nMetadata:", meta)
    print("Text:", doc)
    print("-" * 60)

Chunks added: 1
Total chunks in ChromaDB: 23

Number of Python document chunks: 1

Metadata: {'source': 'python_basics.txt', 'char_count': 265, 'chunk_index': 0}
Text: Python is a high-level programming language known for its simple syntax.

Python is widely used in web development, data science, artificial
intelligence, machine learning and automation.

Popular Python libraries include NumPy, Pandas, Scikit-learn and TensorFlow.
------------------------------------------------------------
